# Ses ile Hayvan Saldırısı Tespiti: Eğitim Defteri

Bu defter, hayvan seslerinden **saldırı/agresyon** tespit eden bir model eğitir.

**Akış:**
1. Google Drive'daki veri setini (VGGSound'dan indirdiğimiz klipler) yükle
2. Örnek sesleri dinle ve spektrogramlarına bak
3. Her klipten **YAMNet** ile ses özellikleri (1024 sayılık "gömü" vektörleri) çıkar
4. Bu özelliklerin üzerine 4 sınıflı küçük bir sinir ağı eğit:
   `saldirgan`, `kurban`, `normal_hayvan`, `arka_plan`
5. Test setinde değerlendir (doğruluk, F1, karışıklık matrisi) ve eğitimsiz YAMNet ile karşılaştır
6. Modeli Drive'a kaydet
7. **Demo:** Bir ses dosyasının zaman çizelgesini çiz ve alarm anlarını göster

Hücreleri yukarıdan aşağıya sırayla çalıştırın (her hücrede ▶ ya da `Shift+Enter`).

## 1. Google Drive'ı bağla

Veri seti Drive'da duruyor. Bu hücre çalışınca Google hesabınıza izin vermeniz istenir.

**Önemli:** Bilgisayarınızda oluşan `veri` klasörünü (içinde `ses` klasörü ve `metadata.csv` var) Drive'da
`MyDrive/hayvan-ses/veri` konumuna yükleyin. Farklı bir yere yüklediyseniz aşağıdaki yolu değiştirin.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

VERI_KLASORU = '/content/drive/MyDrive/hayvan-ses/veri'
CIKTI_KLASORU = '/content/drive/MyDrive/hayvan-ses/model'

## 2. Gerekli paketleri kur

In [ ]:
!pip install -q tensorflow-hub "setuptools<81" librosa soundfile

## 3. Kütüphaneler ve ayarlar

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from IPython.display import Audio, display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

ORNEK_HIZI = 16000          # YAMNet 16 kHz mono ses ister
KARE_ADIMI_SN = 0.48        # YAMNet her 0,48 saniyede bir özellik vektörü üretir
GRUPLAR = ['saldirgan', 'kurban', 'normal_hayvan', 'arka_plan']

os.makedirs(CIKTI_KLASORU, exist_ok=True)
np.random.seed(42)
tf.random.set_seed(42)

## 4. Veri setini yükle

`metadata.csv` her klibin dosya adını, sınıfını (ör. *dog growling*) ve grubunu (ör. *saldirgan*) içerir.

In [ ]:
meta = pd.read_csv(os.path.join(VERI_KLASORU, 'metadata.csv'))
# Dosya yolları Windows'ta oluşturuldu; sadece dosya adını alıp Drive'daki konuma bağlıyoruz
meta['yol'] = meta['dosya'].apply(
    lambda p: os.path.join(VERI_KLASORU, 'ses', os.path.basename(str(p).replace('\\', '/'))))
meta = meta[meta['yol'].apply(os.path.exists)].reset_index(drop=True)
meta['etiket'] = meta['grup'].map({g: i for i, g in enumerate(GRUPLAR)})

print('Toplam klip:', len(meta))
display(meta.groupby(['grup', 'sinif']).size().rename('klip sayısı').to_frame())

meta['grup'].value_counts().reindex(GRUPLAR).plot.bar(color=['tab:red', 'tab:orange', 'tab:green', 'tab:gray'])
plt.title('Gruplara göre klip sayısı'); plt.ylabel('klip'); plt.show()

## 5. Örnek sesleri dinle

Her gruptan bir örnek. Kulaklıkla dinleyip hırlama ile normal havlamanın farkını duyun.

In [ ]:
for g in GRUPLAR:
    ornek = meta[meta['grup'] == g].iloc[0]
    print(f'{g}  ->  {ornek["sinif"]}')
    display(Audio(ornek['yol']))

## 6. Spektrogramlar: Model sesi nasıl "görüyor"?

Spektrogram, sesin zamanla hangi frekanslarda ne kadar güçlü olduğunu gösteren bir resimdir
(yatay eksen zaman, dikey eksen frekans, renk ses gücü). YAMNet sesi önce bu tür bir resme (log-mel spektrogram)
çevirip sonra bir görüntü ağı gibi işler.

In [ ]:
fig, eksenler = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, g in zip(eksenler, GRUPLAR):
    ornek = meta[meta['grup'] == g].iloc[0]
    ses, _ = librosa.load(ornek['yol'], sr=ORNEK_HIZI)
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=ses, sr=ORNEK_HIZI, n_mels=64), ref=np.max)
    ax.imshow(mel, origin='lower', aspect='auto', cmap='magma')
    ax.set_title(f'{g}\n{ornek["sinif"]}'); ax.set_xlabel('zaman'); ax.set_ylabel('mel frekans')
plt.tight_layout(); plt.show()

## 7. YAMNet'i yükle

YAMNet, Google'ın 2 milyon YouTube klibiyle (AudioSet) eğittiği hazır bir ses modelidir.
Biz onu **özellik çıkarıcı** olarak kullanacağız: her 0,48 saniyelik ses parçası için 1024 sayılık bir vektör (gömü/embedding) verir.
Bu vektörler sesin "özetidir"; benzer sesler benzer vektörler üretir. Bu yönteme **transfer öğrenme** denir.

In [ ]:
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
yamnet_siniflari = list(pd.read_csv(yamnet.class_map_path().numpy().decode('utf-8'))['display_name'])
print('YAMNet sınıf sayısı:', len(yamnet_siniflari))

## 8. Her klipten özellik çıkar

Her klip için YAMNet'ten iki şey alıyoruz:
- **gömüler:** (kare sayısı × 1024) matris, eğitimde kullanılacak
- **skorlar:** YAMNet'in kendi 521 sınıfı için ortalama skorları, eğitimsiz karşılaştırma için

Bu adım veri miktarına göre birkaç dakika sürer. Sonuç Drive'a kaydedilir, defteri tekrar çalıştırırsanız yeniden hesaplanmaz.

In [ ]:
onbellek = os.path.join(CIKTI_KLASORU, 'yamnet_ozellikleri.npz')

def ozellik_cikar(yol):
    ses, _ = librosa.load(yol, sr=ORNEK_HIZI, mono=True)
    skor, gomu, _ = yamnet(ses.astype(np.float32))
    return gomu.numpy(), skor.numpy().mean(axis=0)

if os.path.exists(onbellek) and list(np.load(onbellek, allow_pickle=True)['dosyalar']) == list(meta['yol']):
    veri = np.load(onbellek, allow_pickle=True)
    gomuler, yamnet_skorlari = list(veri['gomuler']), veri['skorlar']
    print('Özellikler önbellekten yüklendi.')
else:
    gomuler, skorlar = [], []
    for i, yol in enumerate(meta['yol']):
        g, s = ozellik_cikar(yol)
        gomuler.append(g); skorlar.append(s)
        if (i + 1) % 50 == 0 or i + 1 == len(meta):
            print(f'{i + 1}/{len(meta)} klip işlendi')
    yamnet_skorlari = np.stack(skorlar)
    kutu = np.empty(len(gomuler), dtype=object); kutu[:] = gomuler
    np.savez(onbellek, dosyalar=np.array(meta['yol']), gomuler=kutu, skorlar=yamnet_skorlari)
    print('Özellikler kaydedildi:', onbellek)

print('Örnek klibin gömü boyutu:', gomuler[0].shape)

## 9. Eğitim / doğrulama / test olarak ayır

- **Eğitim (%64):** Model bunlardan öğrenir.
- **Doğrulama (%16):** Eğitim sırasında "ezber yapıyor mu?" diye kontrol için.
- **Test (%20):** Model hiç görmeden en sonda ölçülür. Rapordaki sonuçlar buradan gelir.

Aynı YouTube videosundan gelen klipler hep **aynı** bölümde tutulur (`groups=youtube_id`), böylece test sonucu şişmez.
Her bölümde 4 grubun oranı korunur (`Stratified`).

In [ ]:
ayirici = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
gelistirme_idx, test_idx = next(ayirici.split(meta, meta['etiket'], groups=meta['youtube_id']))
gel = meta.iloc[gelistirme_idx]
e_i, d_i = next(ayirici.split(gel, gel['etiket'], groups=gel['youtube_id']))
egitim_idx, dogrulama_idx = gelistirme_idx[e_i], gelistirme_idx[d_i]

bolum = pd.Series('egitim', index=meta.index)
bolum[dogrulama_idx] = 'dogrulama'; bolum[test_idx] = 'test'
display(pd.crosstab(meta['grup'], bolum).reindex(GRUPLAR))

def kareler(idxler):
    X = np.concatenate([gomuler[i] for i in idxler])
    y = np.concatenate([np.full(len(gomuler[i]), meta['etiket'][i]) for i in idxler])
    return X, y

X_egitim, y_egitim = kareler(egitim_idx)
X_dogrulama, y_dogrulama = kareler(dogrulama_idx)

# Az örneği olan gruplar daha fazla ağırlık alsın (sınıf dengesizliğine karşı)
sayilar = np.bincount(y_egitim, minlength=len(GRUPLAR))
sinif_agirligi = {i: len(y_egitim) / (len(GRUPLAR) * max(n, 1)) for i, n in enumerate(sayilar)}
print('Eğitim karesi:', len(X_egitim), '| Doğrulama karesi:', len(X_dogrulama))
print('Sınıf ağırlıkları:', {GRUPLAR[i]: round(w, 2) for i, w in sinif_agirligi.items()})

## 10. Modeli kur

YAMNet'in 1024 sayılık çıktısının üzerine küçük bir ağ ekliyoruz:

`1024 sayı → 256 nöronluk katman (ReLU) → Dropout %30 → 4 çıktı (Softmax)`

- **ReLU:** Ağın doğrusal olmayan ilişkileri öğrenmesini sağlar.
- **Dropout:** Eğitimde nöronların %30'unu rastgele kapatır, ezberlemeyi azaltır.
- **Softmax:** 4 grubun olasılıklarını verir, toplamları 1'dir.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024,)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(GRUPLAR), activation='softmax'),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 11. Eğit

`EarlyStopping`: Doğrulama kaybı 5 tur boyunca iyileşmezse eğitimi durdurur ve en iyi ağırlıklara geri döner.

In [ ]:
gecmis = model.fit(
    X_egitim, y_egitim,
    validation_data=(X_dogrulama, y_dogrulama),
    epochs=50, batch_size=64, class_weight=sinif_agirligi,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
    verbose=2)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
a1.plot(gecmis.history['loss'], label='eğitim'); a1.plot(gecmis.history['val_loss'], label='doğrulama')
a1.set_title('Kayıp (loss)'); a1.set_xlabel('tur (epoch)'); a1.legend()
a2.plot(gecmis.history['accuracy'], label='eğitim'); a2.plot(gecmis.history['val_accuracy'], label='doğrulama')
a2.set_title('Doğruluk'); a2.set_xlabel('tur (epoch)'); a2.legend()
plt.show()

## 12. Test setinde değerlendir

Her klibin karelerinin olasılıklarını ortalayıp klip için tek bir tahmin üretiyoruz.

- **Accuracy (doğruluk):** Doğru tahmin edilen kliplerin oranı.
- **Precision (kesinlik):** "Saldırgan" dediklerimizin ne kadarı gerçekten saldırgan? Düşükse yanlış alarm çok.
- **Recall (duyarlılık):** Gerçek saldırgan seslerin ne kadarını yakaladık? Düşükse saldırı kaçırıyoruz.
- **F1:** Precision ile recall'un dengeli ortalaması.
- **Karışıklık matrisi:** Satır gerçek grup, sütun tahmin. Köşegen doğrular, diğer hücreler hangi grupların karıştığını gösterir.

In [ ]:
def klip_olasiliklari(idxler):
    X = np.concatenate([gomuler[i] for i in idxler])
    olas = model.predict(X, verbose=0)
    sinirlar = np.cumsum([len(gomuler[i]) for i in idxler])[:-1]
    return np.stack([p.mean(axis=0) for p in np.split(olas, sinirlar)])

test_olas = klip_olasiliklari(test_idx)
y_gercek = meta['etiket'].values[test_idx]
y_tahmin = test_olas.argmax(axis=1)

egitimli_dogruluk = accuracy_score(y_gercek, y_tahmin)
egitimli_f1 = f1_score(y_gercek, y_tahmin, average='macro')
print(f'Test doğruluğu: {egitimli_dogruluk:.3f} | Makro F1: {egitimli_f1:.3f}\n')
print(classification_report(y_gercek, y_tahmin, labels=range(len(GRUPLAR)), target_names=GRUPLAR, zero_division=0))

ConfusionMatrixDisplay(confusion_matrix(y_gercek, y_tahmin, labels=range(len(GRUPLAR))),
                       display_labels=GRUPLAR).plot(cmap='Blues', xticks_rotation=30)
plt.title('Karışıklık matrisi (test)'); plt.show()

## 13. Karşılaştırma: Eğitimsiz YAMNet ne kadar başarılı?

Eğitimin işe yarayıp yaramadığını göstermek için, YAMNet'in kendi sınıflarını (ör. *Growling*, *Screaming*)
elle gruplara eşleyip hiç eğitim yapmadan tahmin ediyoruz. Raporda "eğitim öncesi / sonrası" karşılaştırması olarak kullanılır.

In [ ]:
YAMNET_ESLEME = {
    'saldirgan': ['Growling', 'Roar', 'Hiss', 'Caterwaul'],
    'kurban': ['Screaming', 'Whimper (dog)', 'Yell', 'Crying, sobbing'],
    'normal_hayvan': ['Bark', 'Bow-wow', 'Yip', 'Howl', 'Meow', 'Purr', 'Moo', 'Oink', 'Neigh, whinny',
                      'Crowing, cock-a-doodle-doo', 'Bleat'],
    'arka_plan': ['Speech', 'Children shouting', 'Crowd', 'Car passing by', 'Vehicle', 'Wind', 'Rain', 'Music', 'Guitar'],
}
yamnet_idx = {g: [yamnet_siniflari.index(s) for s in siniflar if s in yamnet_siniflari]
              for g, siniflar in YAMNET_ESLEME.items()}
grup_skor = np.stack([yamnet_skorlari[test_idx][:, yamnet_idx[g]].max(axis=1) for g in GRUPLAR], axis=1)
y_esik_tahmin = grup_skor.argmax(axis=1)

egitimsiz_dogruluk = accuracy_score(y_gercek, y_esik_tahmin)
egitimsiz_f1 = f1_score(y_gercek, y_esik_tahmin, average='macro')
karsilastirma = pd.DataFrame({
    'Doğruluk': [egitimsiz_dogruluk, egitimli_dogruluk],
    'Makro F1': [egitimsiz_f1, egitimli_f1],
}, index=['Eğitimsiz YAMNet (kural)', 'Eğitilmiş model (bizim)']).round(3)
display(karsilastirma)

## 14. Hangi seslerde zorlanıyor?

Test setinde her alt sınıfın (ör. *cat hissing*) ne kadar doğru tahmin edildiği. Raporun "hata analizi" bölümü için.

In [ ]:
test_df = meta.iloc[test_idx][['sinif', 'grup']].copy()
test_df['tahmin'] = [GRUPLAR[i] for i in y_tahmin]
test_df['dogru'] = test_df['grup'] == test_df['tahmin']
display(test_df.groupby(['grup', 'sinif'])['dogru'].agg(['mean', 'count'])
        .rename(columns={'mean': 'doğruluk', 'count': 'test klibi'}).round(2))

## 15. Modeli kaydet

Drive'daki `model` klasörüne üç dosya yazılır:
- `hayvan_ses_modeli.keras`: Keras modeli
- `hayvan_ses_agirliklari.npz`: Aynı modelin ağırlıkları (canlı demo bunu kullanır; Keras sürüm farklarından etkilenmez)
- `ayarlar.json`: Grup adları, alarm eşikleri ve test sonuçları

Canlı demo için bu `model` klasörünü bilgisayarınızdaki `hayvan-ses` klasörüne indirin.

In [ ]:
model.save(os.path.join(CIKTI_KLASORU, 'hayvan_ses_modeli.keras'))

yogun = [k for k in model.layers if isinstance(k, tf.keras.layers.Dense)]
(W1, b1), (W2, b2) = yogun[0].get_weights(), yogun[1].get_weights()
np.savez(os.path.join(CIKTI_KLASORU, 'hayvan_ses_agirliklari.npz'), W1=W1, b1=b1, W2=W2, b2=b2)

ayarlar = {
    'gruplar': GRUPLAR,
    'esik_saldirgan': 0.5,
    'esik_kurban': 0.5,
    'kurban_penceresi_sn': 4.0,
    'test_dogrulugu': round(float(egitimli_dogruluk), 4),
    'test_makro_f1': round(float(egitimli_f1), 4),
    'egitimsiz_yamnet_dogrulugu': round(float(egitimsiz_dogruluk), 4),
}
with open(os.path.join(CIKTI_KLASORU, 'ayarlar.json'), 'w') as f:
    json.dump(ayarlar, f, indent=2, ensure_ascii=False)
print('Kaydedildi:', os.listdir(CIKTI_KLASORU))

## 16. Demo: Bir sesin zaman çizelgesi

Bu fonksiyon bir ses dosyasını baştan sona dinler, her 0,48 saniyede 4 grubun olasılığını hesaplar ve grafiğe döker.
Alarm mantığı canlı demoyla aynıdır:
- **AGRESYON (turuncu):** saldırgan olasılığı ≥ 0,5
- **SALDIRI OLASI (kırmızı):** saldırgan sesten sonraki 4 saniye içinde kurban olasılığı ≥ 0,5

In [ ]:
def alarm_durumlari(olas, zamanlar, a=ayarlar):
    son_saldirgan, durumlar = None, []
    for t, p in zip(zamanlar, olas):
        d = 0
        if p[0] >= a['esik_saldirgan']:
            son_saldirgan, d = t, 1
        if p[1] >= a['esik_kurban'] and son_saldirgan is not None and t - son_saldirgan <= a['kurban_penceresi_sn']:
            d = 2
        durumlar.append(d)
    return np.array(durumlar)

def zaman_analizi(yol, baslik=None):
    ses, _ = librosa.load(yol, sr=ORNEK_HIZI, mono=True)
    _, gomu, _ = yamnet(ses.astype(np.float32))
    olas = model.predict(gomu.numpy(), verbose=0)
    # 3 karelik hareketli ortalama: anlık sıçramaları yumuşatır
    olas = np.stack([np.convolve(olas[:, k], np.ones(3) / 3, mode='same') for k in range(olas.shape[1])], axis=1)
    zamanlar = np.arange(len(olas)) * KARE_ADIMI_SN
    durumlar = alarm_durumlari(olas, zamanlar)

    plt.figure(figsize=(14, 3.5))
    for k, (g, renk) in enumerate(zip(GRUPLAR, ['tab:red', 'tab:orange', 'tab:green', 'tab:gray'])):
        plt.plot(zamanlar, olas[:, k], label=g, color=renk, lw=2)
    for t, d in zip(zamanlar, durumlar):
        if d:
            plt.axvspan(t, t + KARE_ADIMI_SN, color='red' if d == 2 else 'orange', alpha=0.15, lw=0)
    plt.ylim(0, 1); plt.xlabel('zaman (sn)'); plt.ylabel('olasılık')
    plt.title(baslik or os.path.basename(yol)); plt.legend(loc='upper right'); plt.show()

    if (durumlar == 2).any():
        print(f'!!! SALDIRI OLASI: ilk kez {zamanlar[np.argmax(durumlar == 2)]:.1f}. saniyede')
    elif (durumlar == 1).any():
        print(f'AGRESYON sesi: ilk kez {zamanlar[np.argmax(durumlar == 1)]:.1f}. saniyede')
    else:
        print('Alarm yok.')
    display(Audio(ses, rate=ORNEK_HIZI))

# Test setinden her gruptan bir örnek
for g in GRUPLAR:
    secilen = [i for i in test_idx if meta['grup'][i] == g]
    if secilen:
        zaman_analizi(meta['yol'][secilen[0]], f'{g}  ({meta["sinif"][secilen[0]]})')

## 17. Demo: Kendi ses dosyanızı deneyin

Bilgisayarınızdan bir ses ya da video dosyası seçin (wav, mp3, mp4). Örneğin bir YouTube videosunun sesini
kaydedip buraya yükleyebilirsiniz.

In [ ]:
from google.colab import files
yuklenen = files.upload()
for ad in yuklenen:
    zaman_analizi(ad)